# Group Splitting Inference (Unified Config Pipeline)

This notebook runs the unified `inference_pipeline` for the group-splitting model.


## 1) Environment Setup


In [1]:
from pathlib import Path

from pioneerml.integration.zenml import load_step_output
from pioneerml.integration.zenml import utils as zenml_utils
from pioneerml.pipeline.pipelines.inference import inference_pipeline

PROJECT_ROOT = zenml_utils.find_project_root()
zenml_utils.setup_zenml_for_notebook(root_path=PROJECT_ROOT, use_in_memory=True)


Using ZenML repository root: /workspace
Ensure this is the top-level of your repo (.zen must live here).


## 2) Select and Align Inputs

Group-splitting inference needs main event parquet + aligned group-classifier probability parquet.


In [2]:
def _pick_pred(group_pred_dir: Path, main_path: Path) -> Path | None:
    primary = group_pred_dir / f"{main_path.stem}_preds.parquet"
    if primary.exists():
        return primary
    latest = group_pred_dir / f"{main_path.stem}_preds_latest.parquet"
    if latest.exists():
        return latest
    return None


# Main event files
main_dir = Path(PROJECT_ROOT) / "data"
main_paths = sorted(main_dir.glob("ml_output_*.parquet"))

# Optional: uncomment to use fewer files
main_paths = main_paths[:1]

# Group-classifier priors
group_pred_dir = Path(PROJECT_ROOT) / "data" / "group_classifier"
paired = []
for mp in main_paths:
    gp = _pick_pred(group_pred_dir, mp)
    if gp is not None:
        paired.append((str(mp.resolve()), str(gp.resolve())))

if not paired:
    raise RuntimeError(
        "No aligned main/group-classifier prediction pairs found. "
        "Run group-classifier inference first."
    )

parquet_paths = [p[0] for p in paired]
group_probs_parquet_paths = [p[1] for p in paired]

# Model path override (None => auto-resolve latest in trained_models/groupsplitter)
model_path = None

# Output location
output_dir = str((Path(PROJECT_ROOT) / "artifacts" / "inference_small" / "group_splitter").resolve())

print(f"Input files: {len(parquet_paths)}")
print(f"Group prior files: {len(group_probs_parquet_paths)}")


Input files: 1
Group prior files: 1


## 3) Reusable Config Helpers


In [3]:
from pioneerml.plugin.runtime import ensure_plugins_loaded
ensure_plugins_loaded()

from pioneerml_base_plugin.group_splitter.pipeline import load_config
from pioneerml_base_plugin.utils.config_loader import with_loader_sources, with_model_handle_path, with_writer_output


## 4) Build Step Config Blocks


In [4]:
pipeline_config = load_config()["inference"]
pipeline_config = with_loader_sources(
    pipeline_config,
    main_sources=parquet_paths,
    optional_sources_by_name={"group_probs": group_probs_parquet_paths},
)
pipeline_config = with_model_handle_path(pipeline_config, model_path=model_path)
pipeline_config = with_writer_output(
    pipeline_config,
    output_dir=output_dir,
)


## 5) Assemble `pipeline_config` and Run


In [5]:
run = inference_pipeline.with_options(enable_cache=False)(
    pipeline_config=pipeline_config,
)


Initiating a new run for the pipeline: inference_pipeline.
Caching is disabled by default for inference_pipeline.
Using user: default
Using stack: default
  deployer: default
  artifact_store: default
  orchestrator: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step build_model_handle has started.
[build_model_handle] No materializer is registered for type <class 'pioneerml.integration.pytorch.model_handles.torchscript_model_handle.TorchScriptModelHandle'>, so the default Pickle materializer was used. Pickle is not production ready and should only be used for prototyping as the artifacts cannot be loaded when running with a different Python version. Please consider implementing a custom materializer for type <class 'pioneerml.integration.pytorch.model_handles.torchscript_model_handle.TorchScriptModelHandle'> according to the instructions at https://docs.zenml.io/concepts/artifacts/materializers
Step bui

## 6) Inspect Inference Outputs


In [6]:
inference_output = load_step_output(run, "run_inference")
print("inference_output:", inference_output)

predictions_paths = [Path(p) for p in (inference_output.get("predictions_paths") or [])]
if not predictions_paths and inference_output.get("predictions_path"):
    predictions_paths = [Path(inference_output["predictions_path"])]

print("predictions_paths:")
for p in predictions_paths:
    print(" ", p)


inference_output: {'predictions_path': '/workspace/artifacts/inference_small/group_splitter/ml_output_000_preds.parquet', 'predictions_paths': ['/workspace/artifacts/inference_small/group_splitter/ml_output_000_preds.parquet'], 'timestamped_predictions_path': None, 'timestamped_predictions_paths': []}
predictions_paths:
  /workspace/artifacts/inference_small/group_splitter/ml_output_000_preds.parquet


## 7) Optional: Quick Parquet Validation


In [7]:
import gc

import pyarrow as pa
import pyarrow.parquet as pq

if not predictions_paths:
    raise RuntimeError("No prediction parquet files were exported.")

pf = pq.ParquetFile(predictions_paths[0])
print("file:", predictions_paths[0])
print("rows:", pf.metadata.num_rows)
print(pf.schema_arrow)

if pf.num_row_groups > 0:
    sample = pf.read_row_group(0).slice(0, 3)
    print(sample)
else:
    sample = None
    print("No row groups found.")

del sample, pf
gc.collect()
pa.default_memory_pool().release_unused()


file: /workspace/artifacts/inference_small/group_splitter/ml_output_000_preds.parquet
rows: 1024
event_id: int64
pred_hit_pion: list<element: float>
  child 0, element: float
pred_hit_muon: list<element: float>
  child 0, element: float
pred_hit_mip: list<element: float>
  child 0, element: float
time_group_ids: list<element: int64>
  child 0, element: int64
pyarrow.Table
event_id: int64
pred_hit_pion: list<element: float>
  child 0, element: float
pred_hit_muon: list<element: float>
  child 0, element: float
pred_hit_mip: list<element: float>
  child 0, element: float
time_group_ids: list<element: int64>
  child 0, element: int64
----
event_id: [[0,1,2]]
pred_hit_pion: [[[0.99372596,0.9955173,0.9922212,0.99785966,0.9944728,...,0.00009278893,0.00023066744,0.000014928995,0.00001068075,0.0000093404815],[0.9972402,0.99739885,0.99604636,0.9958989,0.99411917,...,0.0036538858,0.006575671,0.007074579,0.004431505,0.010275927],[0.9998572,0.9959486,0.99771994,0.9998379,0.9998615,...,0.0012860413